In [ ]:
# --- CELL A: GUI LAYOUT WITH FIXED COMBINED FEATURES & EXEMPTION TOGGLE ---
import ipywidgets as widgets
import pandas as pd

lbl_style = widgets.Layout(width='125px')
box_style = widgets.Layout(width='100px')
DEDUCTION_DEFAULTS = {'Single': 16100.00, 'Married Filing Jointly': 32200.00}

status_dropdown = widgets.Dropdown(options=['Single', 'Married Filing Jointly'], value='Single', description='Filing Status:')
apply_deduction_check = widgets.Checkbox(value=False, description='Apply Standard Deduction Reduction')
date_picker = widgets.DatePicker(value=pd.to_datetime('2026-06-15').date(), description='Pay Date:')

# ⚖️ NEW ADVISORY TOGGLE: Defaults to True to honor your S-Corp Worker's Comp executive exclusion
nm_wc_exempt_check = widgets.Checkbox(value=True, description="Exempt from NM Workers' Comp Fee (<3 EEs / >10% Owners)", style={'description_width': 'initial'})

# Pair 1: Annual Base Salary
salary_lbl = widgets.Label('Annual Base:', layout=lbl_style)
salary_slide = widgets.IntSlider(value=60000, min=30000, max=250000, step=1000, continuous_update=False)
salary_box = widgets.IntText(value=60000, layout=box_style)
widgets.jslink((salary_slide, 'value'), (salary_box, 'value'))
salary_ui = widgets.HBox([salary_lbl, salary_slide, salary_box])

# Pair 2: Pay Periods
periods_lbl = widgets.Label('Pay Periods:', layout=lbl_style)
periods_slide = widgets.IntSlider(value=12, min=12, max=52, step=1, continuous_update=False)
periods_box = widgets.IntText(value=12, layout=box_style)
widgets.jslink((periods_slide, 'value'), (periods_box, 'value'))
periods_ui = widgets.HBox([periods_lbl, periods_slide, periods_box])

# Pair 3: S-Corp HIP + Payment Type Dropdown
hip_lbl = widgets.Label('S-Corp HIP:', layout=lbl_style)
hip_slide = widgets.FloatSlider(value=1310.0, min=0.0, max=2000.0, step=10.0, continuous_update=False)
hip_box = widgets.FloatText(value=1310.0, layout=box_style)
widgets.jslink((hip_slide, 'value'), (hip_box, 'value'))
hip_toggle = widgets.Dropdown(options=['Direct Paid by ER', 'Reimbursement to EE'], value='Direct Paid by ER', layout=widgets.Layout(width='150px'))
hip_ui = widgets.HBox([hip_lbl, hip_slide, hip_box, hip_toggle])

# Pair 4: S-Corp DIP (Configured to default to $35.50 on Reimbursement to EE)
dip_lbl = widgets.Label('S-Corp DIP:', layout=lbl_style)
dip_slide = widgets.FloatSlider(value=35.50, min=0.0, max=1000.0, step=0.50, continuous_update=False)
dip_box = widgets.FloatText(value=35.50, layout=box_style)
widgets.jslink((dip_slide, 'value'), (dip_box, 'value'))
dip_toggle = widgets.Dropdown(options=['Direct Paid by ER', 'Reimbursement to EE'], value='Reimbursement to EE', layout=widgets.Layout(width='150px'))
dip_ui = widgets.HBox([dip_lbl, dip_slide, dip_box, dip_toggle])

# Pair 5: 401k Deferral
retire_lbl = widgets.Label('401k Def %:', layout=lbl_style)
retire_slide = widgets.FloatSlider(value=0.0, min=0.0, max=25.0, step=0.5, continuous_update=False)
retire_box = widgets.FloatText(value=0.0, layout=box_style)
widgets.jslink((retire_slide, 'value'), (retire_box, 'value'))
retire_ui = widgets.HBox([retire_lbl, retire_slide, retire_box])

# Pair 6: Standard Deduction Override
deduct_lbl = widgets.Label('Std Deduction:', layout=lbl_style)
deduct_slide = widgets.FloatSlider(value=16100.0, min=0.0, max=45000.0, step=100.0, continuous_update=False)
deduct_box = widgets.FloatText(value=16100.0, layout=box_style)
def sync_deduct_slider_to_box(change): deduct_box.value = change['new']
def sync_deduct_box_to_slider(change):
    val = change['new']
    if val is None or str(val).strip() == "": deduct_slide.value = 0.0
    else:
        if val > deduct_slide.max: deduct_slide.max = val * 1.2
        deduct_slide.value = val
deduct_slide.observe(sync_deduct_slider_to_box, names='value')
deduct_box.observe(sync_deduct_box_to_slider, names='value')
deduct_ui = widgets.HBox([deduct_lbl, deduct_slide, deduct_box])

# Pair 7: Flat Additional Federal Withholding (Defaults to $50.00)
fed_extra_lbl = widgets.Label('Addl Fed Tax:', layout=lbl_style)
fed_extra_slide = widgets.FloatSlider(value=50.00, min=0.0, max=1000.0, step=5.0, continuous_update=False)
fed_extra_box = widgets.FloatText(value=50.00, layout=box_style)
widgets.jslink((fed_extra_slide, 'value'), (fed_extra_box, 'value'))
fed_extra_ui = widgets.HBox([fed_extra_lbl, fed_extra_slide, fed_extra_box])

# Pair 8: Flat Additional New Mexico State Withholding (Defaults to $200.00)
nm_extra_lbl = widgets.Label('Addl NM Tax:', layout=lbl_style)
nm_extra_slide = widgets.FloatSlider(value=200.00, min=0.0, max=500.0, step=5.0, continuous_update=False)
nm_extra_box = widgets.FloatText(value=200.00, layout=box_style)
widgets.jslink((nm_extra_slide, 'value'), (nm_extra_box, 'value'))
nm_extra_ui = widgets.HBox([nm_extra_lbl, nm_extra_slide, nm_extra_box])

def on_status_changed(change):
    if deduct_slide.value in [None, 0.0, 16100.0, 32200.0]:
        deduct_slide.value = DEDUCTION_DEFAULTS[change['new']]

status_dropdown.observe(on_status_changed, 'value')
save_button = widgets.Button(description="Commit Pay Run", button_style='success', icon='check')
output_panel = widgets.Output()

# Unified layout presentation includes the new checkbox object seamlessly
input_form = widgets.VBox([status_dropdown, apply_deduction_check, deduct_ui, salary_ui, periods_ui, hip_ui, dip_ui, retire_ui, fed_extra_ui, nm_extra_ui, date_picker, nm_wc_exempt_check, save_button])
display(widgets.HBox([input_form, output_panel]))


In [ ]:
# --- CELL B: MASTER MATH ENGINE & VISUAL DASHBOARD ---
import sqlite3
import matplotlib.pyplot as plt
from IPython.display import display, clear_output

DB_NAME, CSV_BACKUP_NAME, EMPLOYEE_ID = "payroll_ledger.db", "payroll_history_backup.csv", "EE-SCORP-01"
SS_WAGE_BASE, FUTA_WAGE_BASE, NM_SUI_WAGE_BASE = 184500.00, 7000.00, 34800.00
SS_RATE, MEDICARE_RATE, FUTA_NET_RATE, NM_SUI_RATE = 0.062, 0.0145, 0.006, 0.0033

TAX_TABLES_2026 = {
    'Single': {
        'FED': [(12400, 0.10), (50400, 0.12), (105700, 0.22), (201775, 0.24), (256225, 0.32), (640600, 0.35), (float('inf'), 0.37)],
        'NM':  [(8050, 0.00), (13550, 0.015), (20550, 0.032), (24550, 0.032), (33000, 0.043), (41000, 0.043), (58000, 0.047), (74000, 0.047), (102100, 0.047), (116100, 0.049), (217500, 0.049), (331100, 0.049), (float('inf'), 0.059)]
    },
    'Married Filing Jointly': {
        'FED': [(24800, 0.10), (100800, 0.12), (211400, 0.22), (403550, 0.24), (512450, 0.32), (768700, 0.35), (float('inf'), 0.37)],
        'NM':  [(16100, 0.00), (27100, 0.015), (41100, 0.032), (49100, 0.032), (66000, 0.043), (82000, 0.043), (116000, 0.047), (148000, 0.047), (204200, 0.047), (232200, 0.049), (435000, 0.049), (662200, 0.049), (float('inf'), 0.059)]
    }
}

def calc_capped(gross, ytd, cap, rate):
    return 0.0 if ytd >= cap else min(gross, cap - ytd) * rate

def calc_progressive_adjusted(gross, periods, brackets, apply_deduction, deduction_val):
    annual_taxable = max(0.00, (gross * periods) - deduction_val) if apply_deduction else (gross * periods)
    tax_owed = 0.0; prev = 0.0
    for limit, rate in brackets:
        if annual_taxable > limit: tax_owed += (limit - prev) * rate; prev = limit
        else: tax_owed += (annual_taxable - prev) * rate; break
    return tax_owed / periods

# --- 🚀 MASTER CALCULATOR ENGINE (WITH EXEMPTION MATRICES FIXED) 🚀 ---
def calculate_payroll_metrics(status, periods, annual_salary, hip, dip, retire_pct, apply_deduct, deduct_val, fed_extra, nm_extra, nm_wc_exempt, ytd_prior):
    base_gross = annual_salary / periods
    total_payout = base_gross + hip + dip
    deduction_401k = base_gross * (retire_pct / 100.0)
    
    ee_ss = calc_capped(base_gross, ytd_prior, SS_WAGE_BASE, SS_RATE)
    ee_med = base_gross * MEDICARE_RATE
    combined_tax_gross = base_gross + hip + dip - deduction_401k
    
    ee_fed = calc_progressive_adjusted(combined_tax_gross, periods, TAX_TABLES_2026[status]['FED'], apply_deduct, deduct_val) + fed_extra
    ee_nm = calc_progressive_adjusted(combined_tax_gross, periods, TAX_TABLES_2026[status]['NM'], apply_deduct, deduct_val) + nm_extra
    
    # ⚖️ REGULATORY EXEMPTION LAYER RESTORED
    # Statutory Baseline: Employee is $2.25 per run, Employer is $2.55 per run
    nm_wc_fee = 0.00 if nm_wc_exempt else (2.25 * 4) / periods
    er_wc_fee = 0.00 if nm_wc_exempt else (2.55 * 4) / periods
    
    total_ee_taxes = ee_ss + ee_med + ee_fed + ee_nm + nm_wc_fee
    net_pay = total_payout - deduction_401k - total_ee_taxes
    
    er_ss, er_med = ee_ss, ee_med
    er_futa = calc_capped(base_gross, ytd_prior, FUTA_WAGE_BASE, FUTA_NET_RATE)
    er_suta = calc_capped(base_gross, ytd_prior, NM_SUI_WAGE_BASE, NM_SUI_RATE)
    total_er_taxes = er_ss + er_med + er_futa + er_suta + er_wc_fee
    total_company_cost = total_payout + total_er_taxes
    
    return {
        'base_gross': base_gross, 'total_payout': total_payout, 'deduction_401k': deduction_401k,
        'ee_ss': ee_ss, 'ee_med': ee_med, 'ee_fed': ee_fed, 'ee_nm': ee_nm, 'nm_wc_fee': nm_wc_fee, 'er_wc_fee': er_wc_fee,
        'net_pay': net_pay, 'er_futa': er_futa, 'er_suta': er_suta, 'total_er_taxes': total_er_taxes, 'total_company_cost': total_company_cost
    }

def update_dashboard(*args):
    with output_panel:
        clear_output(wait=True)
        conn = sqlite3.connect(DB_NAME); cursor = conn.cursor()
        cursor.execute("""
        CREATE TABLE IF NOT EXISTS payroll_ledger (
            id INTEGER PRIMARY KEY AUTOINCREMENT, employee_id TEXT, pay_period_ending TEXT, 
            gross_wages REAL, hip_wages REAL, dip_wages REAL, pre_tax_401k REAL,
            ee_ss REAL, ee_medicare REAL, ee_fed_tax REAL, ee_nm_tax REAL, ee_wc_fee REAL, er_wc_fee REAL, net_pay REAL,
            er_ss REAL, er_medicare REAL, er_futa REAL, er_nm_sui REAL, total_company_cost REAL, timestamp DATETIME DEFAULT CURRENT_TIMESTAMP
        )""")
        cursor.execute("SELECT SUM(gross_wages - hip_wages - dip_wages) FROM payroll_ledger WHERE employee_id = ?", (EMPLOYEE_ID,))
        res = cursor.fetchone(); ytd_prior = float(res[0]) if res and res[0] is not None else 0.00; conn.close()
        
        # FIXED: Map the nm_wc_exempt_check variable state safely into the calculation engine
        m = calculate_payroll_metrics(
            status_dropdown.value, periods_slide.value, salary_slide.value, 
            hip_slide.value, dip_slide.value, retire_slide.value, 
            apply_deduction_check.value, deduct_slide.value,
            fed_extra_slide.value, nm_extra_slide.value, nm_wc_exempt_check.value, ytd_prior
        )
        
        pure_labor_check = m['base_gross'] - m['deduction_401k'] - (m['ee_ss'] + m['ee_med'] + m['ee_fed'] + m['ee_nm'] + m['nm_wc_fee'])
        reimbursement_cash = sum([hip_slide.value if hip_toggle.value == 'Reimbursement to EE' else 0, dip_slide.value if dip_toggle.value == 'Reimbursement to EE' else 0])
        direct_paid_insurance = sum([hip_slide.value if hip_toggle.value == 'Direct Paid by ER' else 0, dip_slide.value if dip_toggle.value == 'Direct Paid by ER' else 0])
        
        sizes = [max(0, pure_labor_check), m['deduction_401k'], m['ee_fed'], m['ee_nm'], (m['ee_ss'] + m['ee_med']), direct_paid_insurance, reimbursement_cash]
        names = ['Labor Check Base', '401k Def', 'Fed Tax', 'NM Tax', 'FICA taxes', 'ER Direct Insurance', 'EE Cash Reimburse']
        colors = ['#2ecc71', '#3498db', '#e74c3c', '#f1c40f', '#95a5a6', '#9b59b6', '#e67e22']
        combined_labels = [f"{n}\n(${v:,.2f})" for n, v in zip(names, sizes)]
        labels = [l for i, l in enumerate(combined_labels) if sizes[i] > 0]
        colors = [c for i, c in enumerate(colors) if sizes[i] > 0]
        sizes = [s for s in sizes if s > 0]
        
        fig, ax = plt.subplots(figsize=(8.5, 4.0))
        ax.pie(sizes, labels=labels, colors=colors, autopct='%1.1f%%', startangle=140, pctdistance=0.65, labeldistance=1.15, textprops={'fontsize': 9, 'weight': 'bold'})
        ax.set_title("S-Corp Complete Package Allocation", fontsize=11, weight='bold', pad=20)
        plt.tight_layout(); plt.show()
        
        print("=" * 65)
        print("        S-CORP BOOKKEEPING DISBURSEMENT LOG SUMMARY       ")
        print("=" * 65)
        print(f"📄 Total Shareholder Compensation Value Package:  ${(m['total_payout'] - m['deduction_401k'] - (m['ee_ss'] + m['ee_med'] + m['ee_fed'] + m['ee_nm'] + m['nm_wc_fee'])):10,.2f}")
        print("-" * 65)
        print("💵 INDEPENDENT PHYSICAL DISBURSEMENT TRANSACTIONS:")
        print(f"   👉 CHECK #1 (WAGES VOUCHER):               ${pure_labor_check:10,.2f}")
        print(f"   👉 CHECK #2 (REIMBURSEMENT VOUCHER):        ${reimbursement_cash:10,.2f}")
        print(f"   👉 BANK WIRE #3 (DIRECT CARRIER TRANSFER):  ${direct_paid_insurance:10,.2f}")
        print("-" * 65)
        print("💼 EMPLOYER-SIDE STATUTORY OVERHEAD TAX LIABILITIES")
        print(f"   [+] Matching Social Security (6.2%):  ${m['ee_ss']:10,.2f}")
        print(f"   [+] Matching Medicare (1.45%):        ${m['ee_med']:10,.2f}")
        print(f"   [+] Federal Unemployment (FUTA 0.6%): ${m['er_futa']:10,.2f}")
        print(f"   [+] NM State Unemployment (SUTA 0.33%):${m['er_suta']:10,.2f}")
        print(f"   [+] NM Workers' Comp Assessment Match: ${m['er_wc_fee']:10,.2f}")
        print("-" * 65)
        print(f"📊 TOTAL EMPLOYER-SIDE PAYROLL TAXES:   ${m['total_er_taxes']:10,.2f}")
        print(f"🏢 TOTAL TRANSACTION COMPANY COST:       ${m['total_company_cost']:10,.2f}")
        print("=" * 65)

# Connect cell form observers to watch variables live
for w in [status_dropdown, apply_deduction_check, salary_slide, periods_slide, hip_slide, dip_slide, retire_slide, deduct_slide, fed_extra_slide, nm_extra_slide, nm_wc_exempt_check, date_picker, hip_toggle, dip_toggle]:
    w.observe(update_dashboard, 'value')
update_dashboard()


In [ ]:
# --- CELL C: CLEAN DATABASE PERSISTENCE ENGINE ---
def execute_db_append(payload):
    conn = sqlite3.connect(DB_NAME); cursor = conn.cursor()
    cols = "employee_id, pay_period_ending, gross_wages, pre_tax_401k, ee_ss, ee_medicare, ee_fed_tax, ee_nm_tax, ee_wc_fee, net_pay, er_ss, er_medicare, er_futa, er_nm_sui, er_wc_fee, total_company_cost"
    placeholders = ", ".join(["?"] * 16)
    cursor.execute(f"INSERT INTO payroll_ledger ({cols}) VALUES ({placeholders})", payload)
    conn.commit()
    df = pd.read_sql_query("SELECT * FROM payroll_ledger WHERE employee_id = ?", conn, params=[EMPLOYEE_ID])
    df.to_csv(CSV_BACKUP_NAME, index=False); conn.close()

def on_save_clicked(b):
    conn = sqlite3.connect(DB_NAME); cursor = conn.cursor()
    cursor.execute("SELECT SUM(gross_wages) FROM payroll_ledger WHERE employee_id = ?", (EMPLOYEE_ID,))
    res = cursor.fetchone(); ytd_prior = float(res) if res and res is not None else 0.00; conn.close()
    
    # Call the central calculation engine housed in Cell B
    m = calculate_payroll_metrics(
        status_dropdown.value, periods_slide.value, salary_slide.value, 
        hip_slide.value, dip_slide.value, retire_slide.value, 
        apply_deduction_check.value, deduct_slide.value, ytd_prior
    )
    
    # Package values cleanly for the database insert statement
    current_gross = salary_slide.value / periods_slide.value + hip_slide.value + dip_slide.value
    # Update Cell C's SQL string to include: "gross_wages, hip_wages, dip_wages, pre_tax_401k..."
    # And expand your data_payload parameter tuple to store the individual slider snapshots:
    data_payload = (
        EMPLOYEE_ID, str(date_picker.value), current_gross, hip_slide.value, dip_slide.value, m['deduction_401k'], 
        m['ee_ss'], m['ee_med'], m['ee_fed'], m['ee_nm'], m['nm_wc_fee'], m['net_pay'], 
        m['ee_ss'], m['ee_med'], m['er_futa'], m['er_suta'], m['nm_wc_fee'], m['total_company_cost']
    )

    
    execute_db_append(data_payload)
    update_dashboard()
    with output_panel: print(f"\n✅ SUCCESS: Payroll recorded cleanly using Central Math Engine data.")

save_button.on_click(on_save_clicked)
print("Database engine linked directly to centralized master math function.")


In [ ]:
# --- CELL D: LIVE YEAR-END W-2 FORECAST ENGINE (WITH EMPTY/NULL FALLBACK) ---
import sqlite3
import pandas as pd

def run_w2_forecast():
    # Initialize baseline tracking slots at 0.00
    historical_runs = 0
    ytd_401k = ytd_fed_withheld = ytd_nm_withheld = 0.00
    ytd_ss_withheld = ytd_med_withheld = ytd_insurance_premiums = ytd_base_salary_only = 0.00
    
    # 1. SAFELY ATTEMPT TO READ FROM SQLITE DATABASE
    try:
        conn = sqlite3.connect(DB_NAME)
        df = pd.read_sql_query("SELECT * FROM payroll_ledger WHERE employee_id = ?", conn, params=[EMPLOYEE_ID])
        conn.close()
        
        historical_runs = len(df)
        if historical_runs > 0:
            ytd_401k = df['pre_tax_401k'].sum()
            ytd_fed_withheld = df['ee_fed_tax'].sum()
            ytd_nm_withheld = df['ee_nm_tax'].sum()
            ytd_ss_withheld = df['ee_ss'].sum()
            ytd_med_withheld = df['ee_medicare'].sum()
            
            # Check for modern multi-column schema vs legacy tracking loops
            if 'hip_wages' in df.columns:
                ytd_insurance_premiums = df['hip_wages'].sum() + df['dip_wages'].sum()
            else:
                ytd_insurance_premiums = 0.00
            ytd_base_salary_only = df['gross_wages'].sum() - ytd_insurance_premiums
    except (sqlite3.OperationalError, NameError, pd.errors.DatabaseError):
        # 🛡️ AUTOMATED FALLBACK SELECTION TRIGGER: Triggered if database table does not exist yet
        pass

    # 2. EXTRACT ACTIVE SLIDERS TO MODEL REMAINING OR FULL YEAR CHECKS
    total_configured_periods = periods_slide.value
    remaining_periods = max(0, total_configured_periods - historical_runs)
    
    base_salary_per_period = salary_slide.value / total_configured_periods
    hip_period, dip_period = hip_slide.value, dip_slide.value
    deduction_401k_period = base_salary_per_period * (retire_slide.value / 100.0)
    
    future_projected_base_salary = base_salary_per_period * remaining_periods
    future_projected_insurance = (hip_period + dip_period) * remaining_periods
    future_projected_401k = deduction_401k_period * remaining_periods
    
    projected_fed = 0.0
    projected_nm = 0.0
    
    for _ in range(remaining_periods):
        tax_subject_period = base_salary_per_period + hip_period + dip_period - deduction_401k_period
        
        ee_fed = calc_progressive_adjusted(tax_subject_period, total_configured_periods, TAX_TABLES_2026[status_dropdown.value]['FED'], apply_deduction_check.value, deduct_slide.value) + fed_extra_slide.value
        ee_nm = calc_progressive_adjusted(tax_subject_period, total_configured_periods, TAX_TABLES_2026[status_dropdown.value]['NM'], apply_deduction_check.value, deduct_slide.value) + nm_extra_slide.value
        
        projected_fed += ee_fed
        projected_nm += ee_nm

    # 3. COMPILE YEAR-END COMBINED VALUES
    annualized_base_salary = ytd_base_salary_only + future_projected_base_salary
    annualized_insurance_premiums = ytd_insurance_premiums + future_projected_insurance
    annualized_401k_deferred = ytd_401k + future_projected_401k
    
    # S-Corp Tax Rules Mapping Engine
    box_1_wages = annualized_base_salary + annualized_insurance_premiums - annualized_401k_deferred
    box_3_wages = min(annualized_base_salary, SS_WAGE_BASE)
    box_5_wages = annualized_base_salary
    
    final_fed_withheld = ytd_fed_withheld + projected_fed
    final_nm_withheld = ytd_nm_withheld + projected_nm
    final_ss_withheld = box_3_wages * SS_RATE
    final_med_withheld = box_5_wages * MEDICARE_RATE
    
    # 4. RENDER COMPLIANCE DISPLAY PANEL
    print("=" * 65)
    print(f"      FORECASTED 2026 W-2 DISCLOSURE SUMMARY ({EMPLOYEE_ID})      ")
    print("=" * 65)
    if historical_runs == 0:
        print(" ℹ️ STATUS: No committed data found. Running 100% Projection Baseline Model.")
    else:
        print(f" 📦 Tracked Runs inside Ledger: {historical_runs} | 🔮 Remaining Simulated Runs: {remaining_periods}")
    print("-" * 65)
    print(f" 🔹 Box 1 (Wages, Tips, Other Comp):         ${box_1_wages:11,.2f}")
    print(f" 🔹 Box 2 (Federal Income Tax Withheld):      ${final_fed_withheld:11,.2f}")
    print(f" 🔹 Box 3 (Social Security Wages):           ${box_3_wages:11,.2f} (Cap: $184,500)")
    print(f" 🔹 Box 4 (Social Security Tax Withheld):    ${final_ss_withheld:11,.2f}")
    print(f" 🔹 Box 5 (Medicare Wages and Tips):          ${box_5_wages:11,.2f} (No Cap)")
    print(f" 🔹 Box 6 (Medicare Tax Withheld):           ${final_med_withheld:11,.2f}")
    print(f" 🔹 Box 12a (Code D - 401k Deferrals):       ${annualized_401k_deferred:11,.2f}")
    print(f" 🔹 Box 14 (S-CORP OWNER H&W INS):           ${annualized_insurance_premiums:11,.2f}")
    print(f" 🔹 Box 16 (New Mexico State Wages):         ${box_1_wages:11,.2f}")
    print(f" 🔹 Box 17 (New Mexico State Income Tax):     ${final_nm_withheld:11,.2f}")
    print("=" * 65)

# Run the forecast engine
run_w2_forecast()


In [ ]:
# --- CELL E: SECURE DATABASE PURGE & LEVERAGE RESET UTILITY ---
import sqlite3
import os

def purge_payroll_database(execute_wipe=False):
    """Safely drops the ledger table and clears associated CSV flat files."""
    db_file = "payroll_ledger.db"
    csv_file = "payroll_history_backup.csv"
    
    if not execute_wipe:
        print("🛡️ SAFE PREVIEW MODE ACTIVE: No data was altered.")
        print(" -> To completely erase all historical payroll records, execute:")
        print("    purge_payroll_database(execute_wipe=True)")
        return

    # 1. Clear relational SQLite layout
    if os.path.exists(db_file):
        conn = sqlite3.connect(db_file)
        cursor = conn.cursor()
        cursor.execute("DROP TABLE IF EXISTS payroll_ledger")
        conn.commit()
        conn.close()
        print(f"💥 SUCCESS: Relational table dropped from database: '{db_file}'")
    
    # 2. Clear flat CSV backup
    if os.path.exists(csv_file):
        os.remove(csv_file)
        print(f"💥 SUCCESS: Flat backup log removed: '{csv_file}'")
        
    print("\n🔄 SYSTEM RESET COMPLETE: Run Cell A/B to initialize a fresh, empty ledger.")

# Change to True ONLY when you are ready to completely erase all historical logs
purge_payroll_database(execute_wipe=False)


In [ ]:
# --- CELL F: FORM 941 QUARTERLY & NM STATE WITHHOLDING LEDGER ---
import sqlite3
import pandas as pd

def generate_form_941_quarterly_report():
    conn = sqlite3.connect("payroll_ledger.db")
    try:
        df = pd.read_sql_query("SELECT * FROM payroll_ledger WHERE employee_id = 'EE-SCORP-01'", conn)
    except (sqlite3.OperationalError, pd.errors.DatabaseError):
        print("ℹ️ Missing Table. Commit a pay run to view reporting.")
        conn.close(); return
    conn.close()

    if len(df) == 0:
        print("ℹ️ No payroll runs found in the database.")
        return

    df['pay_period_ending'] = pd.to_datetime(df['pay_period_ending'])
    df['Quarter'] = df['pay_period_ending'].dt.to_period('Q')

    print("=" * 65)
    print("   FEDERAL FORM 941 & NM STATE QUARTERLY COMPLIANCE LEDGER  ")
    print("=" * 65)

    for quarter, group in df.groupby('Quarter'):
        q_gross_total = group['gross_wages'].sum()
        q_401k = group['pre_tax_401k'].sum()
        q_fed_withheld = group['ee_fed_tax'].sum()
        q_nm_withheld = group['ee_nm_tax'].sum()
        q_ee_ss = group['ee_ss'].sum()
        q_ee_med = group['ee_medicare'].sum()
        q_ee_wc = group['ee_wc_fee'].sum()
        
        q_hip_total = group['hip_wages'].sum() if 'hip_wages' in group.columns else 0.00
        q_dip_total = group['dip_wages'].sum() if 'dip_wages' in group.columns else 0.00
        q_insurance_total = q_hip_total + q_dip_total
        
        # --- ⚖️ FEDERAL FORM 941 COMPLIANCE VALUES ⚖️ ---
        line_2_wages = q_gross_total - q_401k
        line_3_fit = q_fed_withheld
        line_5a_col1 = q_gross_total - q_insurance_total
        line_5a_col2 = q_ee_ss + q_ee_ss  # Employee + matching Employer
        line_5c_col1 = line_5a_col1
        line_5c_col2 = q_ee_med + q_ee_med  # Employee + matching Employer
        line_5e_col2 = line_5a_col2 + line_5c_col2 
        total_taxes_before_adjustments = line_3_fit + line_5a_col2 + line_5c_col2

        print(f"📅 TAX REPORTING QUARTER: {quarter}")
        print(f" 🔹 Line 2: Wages, Tips, and Other Compensation: ${line_2_wages:11,.2f}")
        print(f" 🔹 Line 3: Federal Income Tax Withheld:        ${line_3_fit:11,.2f}")
        print(f" 🔹 Line 5a: Taxable Social Security Wages:     ${line_5a_col1:11,.2f} | Tax (EE+ER): ${line_5a_col2:,.2f}")
        print(f" 🔹 Line 5c: Taxable Medicare Wages and Tips:   ${line_5c_col1:11,.2f} | Tax (EE+ER): ${line_5c_col2:,.2f}")
        print(f" 🔹 Line 5e: Total SS + Medicare Tax (EE+ER):   ${line_5e_col2:11,.2f}")
        print(f" 🔹 Line 6: Total Taxes Before Adjustments:     ${total_taxes_before_adjustments:11,.2f}")
        print("-" * 65)
        
        # --- 🌶️ NEW MEXICO STATE QUARTERLY COMPLIANCE BREAKOUT (RPD-41285) 🌶️ ---
        print(" 🌶️ NEW MEXICO STATE QUARTERLY REPORTING SUBMISSION:")
        print(f"    -> Taxable Gross State Wages (Box 16 Equivalent): ${line_2_wages:11,.2f}")
        print(f"    -> Total New Mexico Income Tax Withheld:          ${q_nm_withheld:11,.2f}")
        print(f"    -> NM Workers' Comp Fee Collected (Employee):     ${q_ee_wc:11,.2f}")
        # Workers' comp employer assessment match calculation
        nm_er_wc_match = len(group[group['ee_wc_fee'] > 0]) * (2.55 * 4 / periods_slide.value)
        print(f"    -> NM Workers' Comp Fee Match (Employer):         ${nm_er_wc_match:11,.2f}")
        print("-" * 65)
        
        # --- 🏦 UNIFIED MONTHLY LIABILITY TAX TRAFFIC ENGINE 🏦 ---
        print(" 📌 Monthly Tax Deposit Breakdowns (For EFTPS & TAP Portals):")
        
        q_num = quarter.quarter
        year_num = quarter.year
        start_month = (q_num - 1) * 3 + 1
        target_months = [start_month, start_month + 1, start_month + 2]
        
        for line_16_index, m_num in enumerate(target_months, start=1):
            m_name = pd.to_datetime(f"{year_num}-{m_num}-01").strftime('%B')
            m_group = group[group['pay_period_ending'].dt.month == m_num]
            
            fed_liability = 0.00
            nm_tap_withholding = 0.00
            
            if not m_group.empty:
                # Federal Combined Deposit (FIT Withheld + EE FICA + ER FICA)
                fed_liability = m_group['ee_fed_tax'].sum() + (m_group['ee_ss'].sum() * 2) + (m_group['ee_medicare'].sum() * 2)
                # State Monthly Deposit (Pure State Income Tax Withheld)
                nm_tap_withholding = m_group['ee_nm_tax'].sum()
            
            print(f"    -> Month {line_16_index} ({m_name:9}) | 🏛️ EFTPS Federal: ${fed_liability:10,.2f} | 🌶️ NM TAP Portal: ${nm_tap_withholding:10,.2f}")
            
        print(f"    -> Total Quarter Fed Liability Check:     ${total_taxes_before_adjustments:11,.2f}")
        print(f"    -> Total Quarter NM Withholding Check:    ${q_nm_withheld:11,.2f}")
        print("=" * 65)

generate_form_941_quarterly_report()


In [ ]:
# --- CELL G: FORM 940 ANNUAL FUTA & NM SUTA UNEMPLOYMENT MODULE ---
import sqlite3
import pandas as pd

def generate_form_940_annual_report():
    conn = sqlite3.connect("payroll_ledger.db")
    try:
        df = pd.read_sql_query("SELECT * FROM payroll_ledger WHERE employee_id = 'EE-SCORP-01'", conn)
    except (sqlite3.OperationalError, pd.errors.DatabaseError):
        print("ℹ️ Missing Table. Commit a pay run to view reporting.")
        conn.close(); return
    conn.close()
    
    if len(df) == 0:
        print("ℹ️ No annual data detected.")
        return
        
    df['pay_period_ending'] = pd.to_datetime(df['pay_period_ending'])
    df['Year'] = df['pay_period_ending'].dt.year
    
    print("=" * 65)
    print("   ANNUAL FORM 940 FUTA & NM SUI UNEMPLOYMENT COMPLIANCE    ")
    print("=" * 65)
    
    for year, group in df.groupby('Year'):
        total_gross_recorded = group['gross_wages'].sum()
        total_futa_tax_paid = group['er_futa'].sum()
        total_suta_tax_paid = group['er_nm_sui'].sum()
        
        q_hip_total = group['hip_wages'].sum() if 'hip_wages' in group.columns else 0.00
        q_dip_total = group['dip_wages'].sum() if 'dip_wages' in group.columns else 0.00
        total_insurance_exemptions = q_hip_total + q_dip_total
        total_subject_base_wages = total_gross_recorded - total_insurance_exemptions
        
        # --- ⚖️ FEDERAL FUTA ASSESSMENTS ---
        wages_in_excess_of_futa = max(0.00, total_subject_base_wages - FUTA_WAGE_BASE)
        taxable_futa_wages = min(total_subject_base_wages, FUTA_WAGE_BASE)
        
        # --- 🌶️ NEW MEXICO SUTA ASSESSMENTS (NMDWS ES-903A) 🌶️ ---
        # State Unemployment limits: 0.33% up to the first $34,800 of base salary
        wages_in_excess_of_suta = max(0.00, total_subject_base_wages - NM_SUI_WAGE_BASE)
        taxable_suta_wages = min(total_subject_base_wages, NM_SUI_WAGE_BASE)
        
        print(f"📅 TAX COMPLIANCE YEAR: {year}")
        print(f" 🔹 Part 2, Line 3: Total Payments to All Employees:   ${total_gross_recorded:11,.2f}")
        print(f" 🔹 Part 2, Line 4: Payments Exempt from FUTA Tax:    ${total_insurance_exemptions:11,.2f}")
        print("-" * 65)
        print(" 🛠️ FEDERAL UNEMPLOYMENT INSURANCE (FORM 940)")
        print(f"    -> Total Wages in Excess of $7,000 Cap:           ${wages_in_excess_of_futa:11,.2f}")
        print(f"    -> Total Taxable FUTA Wages:                      ${taxable_futa_wages:11,.2f}")
        print(f"    -> Total FUTA Tax Due (@ 0.6%):                   ${total_futa_tax_paid:11,.2f}")
        print("-" * 65)
        print(" 🌶️ NEW MEXICO STATE UNEMPLOYMENT INSURANCE (NMDWS)")
        print(f"    -> Total Wages in Excess of $34,800 Cap:          ${wages_in_excess_of_suta:11,.2f}")
        print(f"    -> Total Taxable NM SUTA Wages:                   ${taxable_suta_wages:11,.2f}")
        print(f"    -> Total NM SUTA Liability Due (@ 0.33%):         ${total_suta_tax_paid:11,.2f}")
        print("=" * 65)

generate_form_940_annual_report()


In [ ]:
# --- CELL H: HISTORICAL QBO DATA SEEDING UTILITY (SHARED SCOPE) ---
import sqlite3
import pandas as pd

# 🛑 CONFIGURATION TARGETS: Change to True only when you are ready to insert
# Note: CELL B must be executed prior to running this cell!
EXECUTE_SEEDING = False

# --- 📋 YOUR EXACT HISTORICAL QUICKBOOKS DATA SETS ---
HISTORICAL_QBO_RUNS = [
    ("2026-01-15", 5000.00, 1026.59, 0.00, 0.00, 747.68, 169.27), # Jan
    ("2026-02-11", 5000.00, 1026.59, 0.00, 0.00, 747.68, 169.27), # Feb
    ("2026-05-07", 5000.00, 0.00, 177.50, 0.00, 610.88, 330.69),   # May
    ("2026-06-15", 5000.00, 1278.64, 35.50, 0.00, 860.94, 382.79),  # Jun
]

def seed_historical_payroll():
    if not EXECUTE_SEEDING:
        print("🛡️ SAFE PREVIEW MODE ACTIVE: No rows inserted.")
        print(" -> Ensure Cell B has been executed first to load global constants.")
        return

    conn = sqlite3.connect(DB_NAME); cursor = conn.cursor()
    
    # Initialize the multi-column table layout on disk
    cursor.execute("""
    CREATE TABLE IF NOT EXISTS payroll_ledger (
        id INTEGER PRIMARY KEY AUTOINCREMENT, employee_id TEXT, pay_period_ending TEXT, 
        gross_wages REAL, hip_wages REAL, dip_wages REAL, pre_tax_401k REAL,
        ee_ss REAL, ee_medicare REAL, ee_fed_tax REAL, ee_nm_tax REAL, ee_wc_fee REAL, net_pay REAL,
        er_ss REAL, er_medicare REAL, er_futa REAL, er_nm_sui REAL, er_wc_fee REAL, total_company_cost REAL, 
        timestamp DATETIME DEFAULT CURRENT_TIMESTAMP
    )""")

    inserted_count = 0
    running_ytd_base_salary = 0.00  # Tracks exact chronological wage base limits
    
    # Sort historically to process chronological limits accurately
    for date_str, base_salary, hip, dip, c_401k, qbo_fed, qbo_nm in sorted(HISTORICAL_QBO_RUNS):
        cursor.execute("SELECT id FROM payroll_ledger WHERE employee_id = ? AND pay_period_ending = ?", (EMPLOYEE_ID, date_str))
        if cursor.fetchone():
            print(f"⚠️ Skipped: Date {date_str} already exists in ledger.")
            running_ytd_base_salary += base_salary
            continue
            
        total_gross = base_salary + hip + dip
        ee_ss = base_salary * SS_RATE
        ee_med = base_salary * MEDICARE_RATE
        nm_wc_fee = (2.55 * 4) / 12
        net_pay = total_gross - c_401k - (ee_ss + ee_med + qbo_fed + qbo_nm + nm_wc_fee)
        
        # --- ⚖️ RUNNING CAP LOGIC (USING GLOBAL VARIABLES FROM CELL B) ⚖️ ---
        # FUTA calculations (0.6% up to first $7k base salary)
        futa_subject_wages = max(0.00, min(base_salary, FUTA_WAGE_BASE - running_ytd_base_salary))
        er_futa = futa_subject_wages * FUTA_NET_RATE
        
        # SUTA calculations (0.33% up to first $34.8k base salary)
        suta_subject_wages = max(0.00, min(base_salary, NM_SUI_WAGE_BASE - running_ytd_base_salary))
        er_suta = suta_subject_wages * NM_SUI_RATE
        
        # Increment the running tracker for the next chronological paycheck loop
        running_ytd_base_salary += base_salary
        
        total_cost = total_gross + ee_ss + ee_med + er_futa + er_suta + nm_wc_fee
        
        insert_sql = """
        INSERT INTO payroll_ledger (
            employee_id, pay_period_ending, gross_wages, hip_wages, dip_wages, pre_tax_401k, 
            ee_ss, ee_medicare, ee_fed_tax, ee_nm_tax, ee_wc_fee, net_pay,
            er_ss, er_medicare, er_futa, er_nm_sui, er_wc_fee, total_company_cost
        ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
        """
        
        cursor.execute(insert_sql, (
            EMPLOYEE_ID, date_str, total_gross, hip, dip, c_401k, ee_ss, ee_med, qbo_fed, qbo_nm, nm_wc_fee, net_pay,
            ee_ss, ee_med, er_futa, er_suta, nm_wc_fee, total_cost
        ))
        inserted_count += 1

    conn.commit()
    df = pd.read_sql_query("SELECT * FROM payroll_ledger WHERE employee_id = ?", conn, params=[EMPLOYEE_ID])
    df.to_csv(CSV_BACKUP_NAME, index=False); conn.close()
    print(f"\n🚀 SUCCESS: {inserted_count} historical paychecks injected cleanly using Cell B parameters!")

seed_historical_payroll()


In [ ]:
# --- CELL I: GENERAL LEDGER JOURNAL ENTRY ENGINE (FULLY SEGREGATED) ---
import sqlite3
import pandas as pd

def generate_payroll_journal_entry():
    conn = sqlite3.connect("payroll_ledger.db")
    try:
        # Fetch the very last committed pay run to generate the matching GL entry
        df = pd.read_sql_query("SELECT * FROM payroll_ledger ORDER BY id DESC LIMIT 1", conn)
    except (sqlite3.OperationalError, pd.errors.DatabaseError):
        print("ℹ️ Missing Table. Commit a payroll run to generate a Journal Entry blueprint.")
        conn.close(); return
    conn.close()

    if len(df) == 0:
        print("ℹ️ No historical payroll logs found. Commit a pay run to view ledger distributions.")
        return

    # Extract transaction values safely from the target database row series
    row = df.iloc[0]
    date_str = row['pay_period_ending']
    gross_total = row['gross_wages']
    ee_ss = row['ee_ss']
    ee_med = row['ee_medicare']
    ee_fed = row['ee_fed_tax']
    ee_nm = row['ee_nm_tax']
    ee_wc = row['ee_wc_fee']
    
    er_futa = row['er_futa']
    er_suta = row['er_nm_sui']
    er_wc_fee = row['er_wc_fee']  # FIXED: Pull directly from row to eliminate duplicate division bugs
    
    # CENTRALIZED SCHEMA SEPARATION: Extract premiums individually from row slots
    true_row_hip = row['hip_wages'] if 'hip_wages' in df.columns else 0.00
    true_row_dip = row['dip_wages'] if 'dip_wages' in df.columns else 0.00
    total_h_and_w_bundle = true_row_hip + true_row_dip
    
    # Mathematically isolate the pure base labor salary operating expense
    pure_salary_expense = gross_total - total_h_and_w_bundle
    
    er_ss_match = ee_ss
    er_med_match = ee_med
    
    # --- TRIPLE-VOUCHER BOOKKEEPING SEGREGATION ---
    # Check #1: Wages Payout (Base Salary less all Employee-side withholdings)
    check_1_wages_payable = pure_salary_expense - (ee_ss + ee_med + ee_fed + ee_nm + ee_wc)
    
    # Check #2 & #3: Disburse cash only if the active toggle matches user reimbursement settings
    # For this standard GL mapping blueprint, we reflect total insurance values as an automated payable
    check_2_hip_reimbursement = true_row_hip
    check_3_dip_reimbursement = true_row_dip

    print("=" * 75)
    print(f"   🏢 DOUBLE-ENTRY PAYROLL GENERAL LEDGER JOURNAL BLUEPRINT (PERIOD: {date_str})  ")
    print("=" * 75)
    print(f" ACCOUNT NAME                                │   DEBIT ($)  │  CREDIT ($) ")
    print("-" * 75)
    # --- DEBITS: Operating Expense Accounts ---
    print(f" 🟢 Shareholder Wages Expense (Base Salary)  │  {pure_salary_expense:10,.2f}  │")
    print(f" 🟢 SCO Health Insurance Premium (HIP) Exp   │  {true_row_hip:10,.2f}  │")
    print(f" 🟢 SCO Dental Insurance Premium (DIP) Exp   │  {true_row_dip:10,.2f}  │")
    print(f"    [Subtotal H&W Benefit Overhead: ${total_h_and_w_bundle:,.2f}]")
    print(f" 🟢 Employer Payroll Taxes (FICA Matching)    │  {(er_ss_match + er_med_match):10,.2f}  │")
    print(f" 🟢 Employer Unemployment Taxes (FUTA + SUTA) │  {(er_futa + er_suta):10,.2f}  │")
    print(f" 🟢 New Mexico Workers' Comp Employer Fee     │  {er_wc_fee:10,.2f}  │")
    print("-" * 75)
    # --- CREDITS: Liability Clearing Accounts ---
    print(f" 🔴 Payroll Liabilities: CHECK #1 (Wages Due)│              │  {check_1_wages_payable:10,.2f}")
    print(f" 🔴 Payroll Liabilities: CHECK #2 (SCO HIP)  │              │  {check_2_hip_reimbursement:10,.2f}")
    print(f" 🔴 Payroll Liabilities: CHECK #3 (SCO DIP)  │              │  {check_3_dip_reimbursement:10,.2f}")
    print(f" 🔴 Payroll Liabilities: Federal FIT Clearing│              │  {ee_fed:10,.2f}")
    print(f" 🔴 Payroll Liabilities: State SIT Clearing  │              │  {ee_nm:10,.2f}")
    print(f" 🔴 Payroll Liabilities: FICA Tax Clearing   │              │  {((ee_ss + er_ss_match) + (ee_med + er_med_match)):10,.2f}")
    print(f" 🔴 Payroll Liabilities: FUTA Tax Clearing   │              │  {er_futa:10,.2f}")
    print(f" 🔴 Payroll Liabilities: NM SUTA Tax Clearing│              │  {er_suta:10,.2f}")
    print(f" 🔴 Payroll Liabilities: NM WC Fee Clearing  │              │  {(ee_wc + er_wc_fee):10,.2f}")
    print("-" * 75)
    
    # Audit Validation Step
    total_debits = pure_salary_expense + true_row_hip + true_row_dip + er_ss_match + er_med_match + er_futa + er_suta + er_wc_fee
    total_credits = check_1_wages_payable + check_2_hip_reimbursement + check_3_dip_reimbursement + ee_fed + ee_nm + (ee_ss + er_ss_match) + (ee_med + er_med_match) + er_futa + er_suta + ee_wc + er_wc_fee
    print(f" ✅ AUDIT TRIAL BALANCE TOTALS:              │  {total_debits:10,.2f}  │  {total_credits:10,.2f} ")
    print("=" * 75)

generate_payroll_journal_entry()
